# Sample Project Notebook

Goal of this project: simulate a 1m radius ball of Uranium in a box of air. represent the following tallies: 
- nu-fission: how many fission reactions occur
- absorption: how many absorption reactions occur
- sc

First, create the materials.

In [2]:
import openmc
import matplotlib.pyplot as plt
"Notes:"
"-  units are in seconds, centimeters, and electronvolts."

'-  units are in seconds, centimeters, and electronvolts.'

In [3]:
uranium = openmc.Material(name="uranium")
uranium.add_nuclide("U238", .19, 'wo')
uranium.add_nuclide("U235", .8, 'wo')
uranium.add_nuclide("U234", .01, 'wo')
uranium.set_density("g/cm3", 19)

air = openmc.Material(name="air")
air.add_element("N", 0.78)
air.add_element("O", 0.22)
air.set_density("g/cm3", 0.001205)

materials = openmc.Materials([uranium, air])

export materials to xml file: 

In [4]:
materials.export_to_xml()

Second, create geometries: 

In [5]:
sphere = openmc.Sphere(r=1)
inside_sphere = -sphere
outside_sphere = +sphere

zregion = -openmc.ZPlane(5000, boundary_type="vacuum") & +openmc.ZPlane(-5000, boundary_type="vacuum")
yregion = -openmc.YPlane(5000, boundary_type="vacuum") & +openmc.YPlane(-5000, boundary_type="vacuum")
xregion = -openmc.XPlane(5000, boundary_type="vacuum") & +openmc.XPlane(-5000, boundary_type="vacuum")

inside_box = outside_sphere & zregion & yregion & xregion

# step 3: combine materials with geometry
uranium_ball = openmc.Cell()
uranium_ball.fill = uranium
uranium_ball.region = inside_sphere

box = openmc.Cell()
box.region = inside_box
box.fill = air

geometry = openmc.Geometry([uranium_ball, box])

Export geometries to xml file: 

In [6]:
geometry.export_to_xml()

Third, assign settings. These determine how many neutrons are coming from the source, and where the source is located in the model. 

In [7]:
settings = openmc.Settings()
settings.run_mode = "fixed source"
settings.particles = 10000
settings.batches = 100

source = openmc.IndependentSource()
source.space = openmc.stats.Point((0,0,0))
source.strength = 0.5
settings.source = source

Optionally, add surface filters that determine current. 

In [8]:
surfacefilter = openmc.SurfaceFilter(sphere)
cellfilter = openmc.CellFilter([box, uranium_ball])

Fourth, create tallies. these are counts of reactions that happen in a certain material. 

In [9]:
current_tally = openmc.Tally(name="current_tally")
current_tally.filters = [surfacefilter]
current_tally.scores = ['current']

flux_absorption_tally = openmc.Tally(name="flux_absorption_tally")
flux_absorption_tally.filters = [cellfilter]
flux_absorption_tally.scores = ['flux', 'absorption', 'nu-fission']

tallies_object = openmc.Tallies([flux_absorption_tally, current_tally])

Export settings and tallies to xml. 

In [10]:
tallies_object.export_to_xml()
settings.export_to_xml()

### return 1: xml sheets of everything. 

In [11]:
openmc.run()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

### return 2: create an image of the model 

In [14]:
plot = openmc.Plot()
plot.basis = 'xz'  # or 'xy', 'yz'
plot.origin = (0, 0, 0)
plot.width = (20, 20)  # cm — zoom in near your sphere, not the full 10m box
plot.pixels = (800, 800)
plot.color_by = 'material'

plots = openmc.Plots([plot])
plots.export_to_xml()
openmc.plot_geometry()

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################